# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ahmedshereef1/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Baseline rule

I prioritize content for review when it is both stale and has meaningful search demand.

The baseline gives a higher score to content that has not been updated recently
and has higher search volume. The purpose is to create a simple and transparent
priority queue for content review.

The rule is a directional decision-support baseline. It does not claim to
predict Google's ranking algorithm.

### Signals

1. **Staleness**
   - Signal: `days_since_last_update`
   - FlyRank connection: refresh/staleness logic
   - Higher values indicate older content.

2. **Search volume**
   - Signal: `search_volume`
   - FlyRank connection: opportunity / quick-win logic
   - Higher values indicate greater search demand.

### Reason codes

- `STALE_HIGH_VOLUME`: content is stale and has high search demand.
- `STALE_CONTENT`: content is stale but does not have high search demand.
- `HIGH_SEARCH_VOLUME`: content has high search demand but is not classified as stale.
- `LOW_PRIORITY`: neither signal reaches the baseline threshold.

In [11]:
import pandas as pd
import numpy as np
from pathlib import Path

# ============================================================
# 1. LOAD DATA
# ============================================================

DATA_PATH = Path("../../data/raw/content_refresh_anonymized.csv")

# Fallback for Colab / repo-root execution
if not DATA_PATH.exists():
    DATA_PATH = Path("data/raw/content_refresh_anonymized.csv")

df = pd.read_csv(DATA_PATH)

print(f"Dataset shape: {df.shape}")
print(f"Number of columns: {len(df.columns)}")

# Required columns for this baseline
required_columns = [
    "content_id",
    "search_volume",
    "days_since_last_update",
]

missing = [c for c in required_columns if c not in df.columns]

if missing:
    raise ValueError(f"Missing required columns: {missing}")

print("Required columns found.")


# ============================================================
# SIGNAL 1 — STALENESS
# Linked to FlyRank refresh/staleness logic
# ============================================================

staleness_bins = [-1, 30, 60, 90, 120, 180, 365, np.inf]

staleness_labels = [
    "0-30",
    "31-60",
    "61-90",
    "91-120",
    "121-180",
    "181-365",
    ">365",
]

df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=staleness_bins,
    labels=staleness_labels
)

staleness_table = (
    df["staleness_bucket"]
    .value_counts(dropna=False)
    .sort_index()
    .rename("n")
    .to_frame()
)

print("\n" + "=" * 60)
print("SIGNAL 1: STALENESS")
print("=" * 60)

display(staleness_table)


# ============================================================
# SIGNAL 2 — SEARCH VOLUME
# Linked to FlyRank quick-win / opportunity logic
# ============================================================

volume_bins = [-np.inf, 0, 10, 50, 100, 500, 1000, np.inf]

volume_labels = [
    "0",
    "1-10",
    "11-50",
    "51-100",
    "101-500",
    "501-1000",
    ">1000",
]

df["volume_bucket"] = pd.cut(
    df["search_volume"],
    bins=volume_bins,
    labels=volume_labels
)

volume_table = (
    df["volume_bucket"]
    .value_counts(dropna=False)
    .sort_index()
    .rename("n")
    .to_frame()
)

print("\n" + "=" * 60)
print("SIGNAL 2: SEARCH VOLUME")
print("=" * 60)

display(volume_table)


# ============================================================
# SIGNAL VERDICTS
# ============================================================

# Summarize the observed distribution of each signal.
# The bucket tables show that both signals have meaningful variation.
# We use MIXED because these distributions alone do not establish
# that either signal guarantees a successful content action.

staleness_summary = (
    df.groupby("staleness_bucket", observed=True)["days_since_last_update"]
    .agg(["count", "median"])
    .reset_index()
)

volume_summary = (
    df.groupby("volume_bucket", observed=True)["search_volume"]
    .agg(["count", "median"])
    .reset_index()
)

print("\n" + "=" * 60)
print("SIGNAL VERDICTS")
print("=" * 60)

print("\nStaleness summary:")
display(staleness_summary)

print("\nSearch-volume summary:")
display(volume_summary)

staleness_verdict = "MIXED"
volume_verdict = "MIXED"

print("\nVerdict for staleness:", staleness_verdict)
print("Verdict for search volume:", volume_verdict)


# ============================================================
# RULE
# ============================================================

# Use observed quantiles rather than arbitrary absolute values.
# This keeps the baseline tied to the actual starter dataset.

STALE_THRESHOLD = df["days_since_last_update"].quantile(0.75)
HIGH_VOLUME_THRESHOLD = df["search_volume"].quantile(0.75)

print("\nBaseline thresholds:")
print(f"Stale threshold: {STALE_THRESHOLD:.2f} days")
print(f"High-volume threshold: {HIGH_VOLUME_THRESHOLD:.2f}")


# Score:
# +2 = stale
# +1 = high search volume

df["baseline_score"] = (
    (df["days_since_last_update"] >= STALE_THRESHOLD).astype(int) * 2
    + (df["search_volume"] >= HIGH_VOLUME_THRESHOLD).astype(int)
)


# ============================================================
# ONE REASON CODE
# ============================================================

df["reason_code"] = np.select(
    [
        (
            (df["days_since_last_update"] >= STALE_THRESHOLD)
            &
            (df["search_volume"] >= HIGH_VOLUME_THRESHOLD)
        ),

        df["days_since_last_update"] >= STALE_THRESHOLD,

        df["search_volume"] >= HIGH_VOLUME_THRESHOLD,
    ],
    [
        "STALE_HIGH_VOLUME",
        "STALE_CONTENT",
        "HIGH_SEARCH_VOLUME",
    ],
    default="LOW_PRIORITY",
)


# ============================================================
# ACTION LABEL
# ============================================================

df["action_label"] = np.select(
    [
        df["reason_code"] == "STALE_HIGH_VOLUME",
        df["reason_code"] == "STALE_CONTENT",
        df["reason_code"] == "HIGH_SEARCH_VOLUME",
    ],
    [
        "Refresh content",
        "Review freshness",
        "Review opportunity",
    ],
    default="Monitor",
)


# Preview
print("\n" + "=" * 60)
print("BASELINE PREVIEW")
print("=" * 60)

display(
    df[
        [
            "content_id",
            "days_since_last_update",
            "search_volume",
            "baseline_score",
            "reason_code",
            "action_label",
        ]
    ].head(20)
)

Dataset shape: (30000, 44)
Number of columns: 44
Required columns found.

SIGNAL 1: STALENESS


,n
staleness_bucket,
0-30,20480
31-60,128
61-90,47
91-120,9115
121-180,56
181-365,169
>365,5



SIGNAL 2: SEARCH VOLUME


,n
volume_bucket,
0,11081
1-10,7311
11-50,4989
51-100,1102
101-500,2021
501-1000,468
>1000,560
NaN,2468



SIGNAL VERDICTS

Staleness summary:


,staleness_bucket,count,median
0,0-30,20480,20.0
1,31-60,128,41.0
2,61-90,47,89.0
3,91-120,9115,104.0
4,121-180,56,151.0
5,181-365,169,211.0
6,>365,5,373.0



Search-volume summary:


,volume_bucket,count,median
0,0,11081,0.0
1,1-10,7311,10.0
2,11-50,4989,30.0
3,51-100,1102,70.0
4,101-500,2021,210.0
5,501-1000,468,720.0
6,>1000,560,2900.0



Verdict for staleness: MIXED
Verdict for search volume: MIXED

Baseline thresholds:
Stale threshold: 104.00 days
High-volume threshold: 20.00

BASELINE PREVIEW


,content_id,days_since_last_update,search_volume,baseline_score,reason_code,action_label
0,content_304f48230142,20,10.0,0,LOW_PRIORITY,Monitor
1,content_a1fb4e703a9e,25,90.0,1,HIGH_SEARCH_VOLUME,Review opportunity
2,content_9aa793d4d895,20,0.0,0,LOW_PRIORITY,Monitor
3,content_331d6c4de07b,22,10.0,0,LOW_PRIORITY,Monitor
4,content_d99b7a2d90ca,14,0.0,0,LOW_PRIORITY,Monitor
5,content_d4084a4bc775,20,720.0,1,HIGH_SEARCH_VOLUME,Review opportunity
6,content_9a34b442b552,20,0.0,0,LOW_PRIORITY,Monitor
7,content_a63219c6e95a,22,590.0,1,HIGH_SEARCH_VOLUME,Review opportunity
8,content_5e6c160719bc,20,0.0,0,LOW_PRIORITY,Monitor
9,content_c27558df2b0c,104,0.0,2,STALE_CONTENT,Review freshness


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [12]:
# ============================================================
# 2. BUILD THE RANKED QUEUE
# ============================================================

ranked = (
    df.sort_values(
        by=[
            "baseline_score",
            "search_volume",
            "days_since_last_update",
        ],
        ascending=[
            False,
            False,
            False,
        ],
        na_position="last",
    )
    .reset_index(drop=True)
)

ranked["rank"] = ranked.index + 1


# ============================================================
# SAVE OUTPUT
# ============================================================

output_path = Path("work/outputs/baseline_action_score.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

ranked.to_csv(
    output_path,
    index=False
)

print(f"Saved ranked queue to: {output_path}")
print(f"Number of rows: {len(ranked)}")


# ============================================================
# SHOW TOP 20
# ============================================================

queue_columns = [
    "rank",
    "content_id",
    "search_volume",
    "days_since_last_update",
    "baseline_score",
    "reason_code",
    "action_label",
]

display(
    ranked[queue_columns].head(20)
)

Saved ranked queue to: work\outputs\baseline_action_score.csv
Number of rows: 30000


,rank,content_id,search_volume,days_since_last_update,baseline_score,reason_code,action_label
0,1,content_ef99c4abd9ab,74000.0,104,3,STALE_HIGH_VOLUME,Refresh content
1,2,content_bf67a444faef,60500.0,104,3,STALE_HIGH_VOLUME,Refresh content
2,3,content_5ec29ae79c60,60500.0,104,3,STALE_HIGH_VOLUME,Refresh content
3,4,content_deb54e9e19cd,60500.0,104,3,STALE_HIGH_VOLUME,Refresh content
4,5,content_454cc6654c6e,60500.0,104,3,STALE_HIGH_VOLUME,Refresh content
5,6,content_7868341d97dd,40500.0,104,3,STALE_HIGH_VOLUME,Refresh content
6,7,content_84fe9d0a707a,40500.0,104,3,STALE_HIGH_VOLUME,Refresh content
7,8,content_6b41450ae50c,27100.0,104,3,STALE_HIGH_VOLUME,Refresh content
8,9,content_e7eb94e121b9,22200.0,104,3,STALE_HIGH_VOLUME,Refresh content
9,10,content_c861e30f2f7a,22200.0,104,3,STALE_HIGH_VOLUME,Refresh content


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [13]:
# ============================================================
# 3. TOP-20 REVIEW
# ============================================================

top20 = ranked.head(20).copy()

# Confidence note based on the baseline score
top20["confidence_note"] = np.select(
    [
        top20["baseline_score"] == 3,
        top20["baseline_score"] == 2,
        top20["baseline_score"] == 1,
        top20["baseline_score"] == 0,
    ],
    [
        "Moderate: both selected signals are present.",
        "Moderate: staleness is the main prioritization signal.",
        "Lower: search demand is present without strong staleness.",
        "Low: neither signal strongly supports prioritization.",
    ],
    default="Low",
)

# Explain what could make each recommendation wrong
top20["what_would_make_it_wrong"] = np.select(
    [
        top20["reason_code"] == "STALE_HIGH_VOLUME",
        top20["reason_code"] == "STALE_CONTENT",
        top20["reason_code"] == "HIGH_SEARCH_VOLUME",
        top20["reason_code"] == "LOW_PRIORITY",
    ],
    [
        "The page may still satisfy search intent despite being old, "
        "or high search demand may not represent a useful refresh opportunity.",

        "The page may be intentionally unchanged, so age alone may not "
        "justify a refresh.",

        "High search demand does not by itself show that the content "
        "needs improvement or updating.",

        "The two selected signals may miss an opportunity that requires "
        "other content or performance evidence.",
    ],
    default="The simple two-signal rule may not capture the full context.",
)

top20_review = top20[
    [
        "rank",
        "content_id",
        "search_volume",
        "days_since_last_update",
        "baseline_score",
        "reason_code",
        "action_label",
        "confidence_note",
        "what_would_make_it_wrong",
    ]
]

display(top20_review)

,rank,content_id,search_volume,days_since_last_update,baseline_score,reason_code,action_label,confidence_note,what_would_make_it_wrong
0,1,content_ef99c4abd9ab,74000.0,104,3,STALE_HIGH_VOLUME,Refresh content,Moderate: both selected signals are present.,The page may still satisfy search intent despi...
1,2,content_bf67a444faef,60500.0,104,3,STALE_HIGH_VOLUME,Refresh content,Moderate: both selected signals are present.,The page may still satisfy search intent despi...
2,3,content_5ec29ae79c60,60500.0,104,3,STALE_HIGH_VOLUME,Refresh content,Moderate: both selected signals are present.,The page may still satisfy search intent despi...
3,4,content_deb54e9e19cd,60500.0,104,3,STALE_HIGH_VOLUME,Refresh content,Moderate: both selected signals are present.,The page may still satisfy search intent despi...
4,5,content_454cc6654c6e,60500.0,104,3,STALE_HIGH_VOLUME,Refresh content,Moderate: both selected signals are present.,The page may still satisfy search intent despi...
5,6,content_7868341d97dd,40500.0,104,3,STALE_HIGH_VOLUME,Refresh content,Moderate: both selected signals are present.,The page may still satisfy search intent despi...
6,7,content_84fe9d0a707a,40500.0,104,3,STALE_HIGH_VOLUME,Refresh content,Moderate: both selected signals are present.,The page may still satisfy search intent despi...
7,8,content_6b41450ae50c,27100.0,104,3,STALE_HIGH_VOLUME,Refresh content,Moderate: both selected signals are present.,The page may still satisfy search intent despi...
8,9,content_e7eb94e121b9,22200.0,104,3,STALE_HIGH_VOLUME,Refresh content,Moderate: both selected signals are present.,The page may still satisfy search intent despi...
9,10,content_c861e30f2f7a,22200.0,104,3,STALE_HIGH_VOLUME,Refresh content,Moderate: both selected signals are present.,The page may still satisfy search intent despi...


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [14]:
# ============================================================
# WEAK PICKS
# ============================================================

weak_picks = ranked[
    ranked["baseline_score"] == 1
].head(10).copy()

weak_picks["why_weak"] = (
    "High search volume creates an opportunity signal, but the "
    "content is not classified as stale. Search demand alone does "
    "not prove that a refresh is needed."
)

display(
    weak_picks[
        [
            "rank",
            "content_id",
            "search_volume",
            "days_since_last_update",
            "baseline_score",
            "reason_code",
            "action_label",
            "why_weak",
        ]
    ]
)

# ============================================================
# LEAKAGE CHECK
# ============================================================

print("\n" + "=" * 60)
print("LEAKAGE CHECK")
print("=" * 60)

baseline_inputs = [
    "days_since_last_update",
    "search_volume",
]

print("Baseline input features:")
for feature in baseline_inputs:
    print(f"  ✓ {feature}")


# Columns that must NOT be used to calculate the score
forbidden_inputs = [
    "content_id",
    "client_id",
]

print("\nIdentity/client fields excluded from scoring:")
for feature in forbidden_inputs:
    if feature in df.columns:
        print(f"  ✓ {feature} not used")


print("\nLeakage conclusion:")
print("✓ No target/label-derived feature was used.")
print("✓ No future-window feature was used by the score.")
print("✓ No product flag was used as a scoring input.")
print("✓ The score uses only the two selected observed signals.")

print(
    "\nThe baseline is a directional decision-support rule, "
    "not a predictive model."
)

,rank,content_id,search_volume,days_since_last_update,baseline_score,reason_code,action_label,why_weak
9091,9092,content_cd6760921db8,49500.0,41,1,HIGH_SEARCH_VOLUME,Review opportunity,High search volume creates an opportunity sign...
9092,9093,content_f76ccf7a7834,49500.0,22,1,HIGH_SEARCH_VOLUME,Review opportunity,High search volume creates an opportunity sign...
9093,9094,content_83e3da1394ac,49500.0,22,1,HIGH_SEARCH_VOLUME,Review opportunity,High search volume creates an opportunity sign...
9094,9095,content_ee4630879d03,49500.0,20,1,HIGH_SEARCH_VOLUME,Review opportunity,High search volume creates an opportunity sign...
9095,9096,content_8ca50876b0df,40500.0,26,1,HIGH_SEARCH_VOLUME,Review opportunity,High search volume creates an opportunity sign...
9096,9097,content_c841193dc692,40500.0,26,1,HIGH_SEARCH_VOLUME,Review opportunity,High search volume creates an opportunity sign...
9097,9098,content_eb1510f4b5f1,33100.0,41,1,HIGH_SEARCH_VOLUME,Review opportunity,High search volume creates an opportunity sign...
9098,9099,content_19bdaa296a9b,33100.0,22,1,HIGH_SEARCH_VOLUME,Review opportunity,High search volume creates an opportunity sign...
9099,9100,content_6f6a4e56098c,33100.0,20,1,HIGH_SEARCH_VOLUME,Review opportunity,High search volume creates an opportunity sign...
9100,9101,content_f04ea036f597,33100.0,14,1,HIGH_SEARCH_VOLUME,Review opportunity,High search volume creates an opportunity sign...



LEAKAGE CHECK
Baseline input features:
  ✓ days_since_last_update
  ✓ search_volume

Identity/client fields excluded from scoring:
  ✓ content_id not used
  ✓ client_id not used

Leakage conclusion:
✓ No target/label-derived feature was used.
✓ No future-window feature was used by the score.
✓ No product flag was used as a scoring input.
✓ The score uses only the two selected observed signals.

The baseline is a directional decision-support rule, not a predictive model.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.